In [0]:
dbutils.widgets.text("base_path", "/Volumes/swiggy/pipeline/raw_data")

In [0]:
BASE_PATH = dbutils.widgets.get("base_path")

LANDING_ORDERS = BASE_PATH + "/landing/orders"
CHECKPOINT_ORDERS = BASE_PATH + "/checkpoints/orders"
BRONZE_ORDERS = BASE_PATH + "/bronze/orders"

print(f"BASE_PATH       : {BASE_PATH}")
print(f"LANDING_ORDERS  : {LANDING_ORDERS}")
print(f"CHECKPOINT_ORDERS: {CHECKPOINT_ORDERS}")
print(f"BRONZE_ORDERS   : {BRONZE_ORDERS}")

In [0]:
from pyspark.sql.types import StructField , StructType , StringType , DoubleType , IntegerType , BooleanType

order_schema = StructType([
  StructField("order_id" , StringType() , False),
  StructField("customer_id" , StringType() , False),
  StructField("restaurant_id" , StringType() , False),
  StructField("order_value" , DoubleType() , False),
  StructField("item_count" , IntegerType() , False),
  StructField("status" , StringType() , False),
  StructField("city" , StringType() , False),
  StructField("placed_at" , StringType() , True),
  StructField("payment_mode" , StringType() , True),
  StructField("discount_applied" , BooleanType() , True),
  StructField("discount_amount" , DoubleType() , True)
])
  


In [0]:
raw_df = (spark.readStream  #streamread not one time read
          .format("cloudfiles") #autoloader
          .option("cloudfiles.format" , "json") #as files are in json
          .option("cloudfiles.schemalocation" , CHECKPOINT_ORDERS + "/schema") #stores schema info
          .schema(order_schema) #my explicit schema
          .load(LANDING_ORDERS) #read from here
          )
from pyspark.sql.functions import current_timestamp , current_date , col
bronze_df = (
    raw_df
    .withColumn("bronze_ingested_at",  current_timestamp())
    .withColumn("bronze_ingest_date",  current_date())
    .withColumn("bronze_source_file",  col("_metadata.file_path"))
)

print("Schema ready")

In [0]:
(
    bronze_df.writeStream              # streaming write — matches readStream
              .format("delta")         # write as Delta table
              .outputMode("append")    # always add new rows, never overwrite
              .option("checkpointLocation", CHECKPOINT_ORDERS)  # remember progress
              .trigger(availableNow=True)   # process all files now then stop
              .start(BRONZE_ORDERS)         # write to this path
              .awaitTermination()           # wait until done before next cell runs
)


In [0]:
bronze_df_check = spark.read.format("delta").load(BRONZE_ORDERS)
print(f"Total rows: {bronze_df_check.count()}")
bronze_df_check.show(5, truncate=False)
bronze_df_check.printSchema()